In [6]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader

In [7]:
from gensim.models import KeyedVectors

model = KeyedVectors.load("glove50.kv")

In [8]:
from langchain.embeddings.base import Embeddings
import numpy as np

class GensimEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return [self._embed(text) for text in texts]

    def embed_query(self, text):
        return self._embed(text)

    def _embed(self, text):
        words = text.split()
        vectors = [self.model[w] for w in words if w in self.model]
        
        if len(vectors) == 0:
            return [0.0] * self.model.vector_size
        
        return np.mean(vectors, axis=0).tolist()

In [9]:
embedding=GensimEmbeddings(model)

In [12]:
loader=TextLoader("data.txt")

In [13]:
text=loader.load()

In [17]:
text

[Document(metadata={'source': 'data.txt'}, page_content='I love learning machine learning. , general\nArtificial intelligence is the future. , general\nChatGPT helps in coding and problem solving. , general\nPython is a powerful programming language. , general\nData science involves statistics and programming. , general\nDeep learning is a subset of machine learning. , general\nNatural language processing deals with text data. , general\n\nI really enjoyed this movie, it was fantastic! , positive\nThe product quality is terrible and disappointing. , negative\nAmazing experience, I would definitely recommend it. , positive\nWorst service ever, I am not happy at all. , negative\nThe food was okay, not great but not bad either. , neutral\nAbsolutely loved it! , positive\nI hate this app, it crashes every time. , negative\n\nUser: Hi, how are you? , conversation\nBot: I am fine, how can I help you? , conversation\nUser: What is machine learning? , conversation\nBot: Machine learning is a f

In [20]:
##splitter 
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [24]:
splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)

In [26]:
docs=splitter.split_documents(text)

In [33]:
vector_db=Chroma.from_documents(docs,embedding)

In [34]:
vector_db

In [36]:
top_k=vector_db.similarity_search("what is ai")

In [38]:
top_k[0].page_content

'Worst service ever, I am not happy at all. , negative\nThe food was okay, not great but not bad either. , neutral\nAbsolutely loved it! , positive\nI hate this app, it crashes every time. , negative'

In [39]:
retriever=vector_db.as_retriever()

In [41]:
top_m_by_retriever=retriever.invoke("what is machine learning")

In [42]:
vector_embed=embedding.embed_query("what is machine learning")

In [44]:
len(vector_embed)

50

In [48]:
se=vector_db.similarity_search_by_vector(vector_embed)

In [49]:
se

[Document(id='4ed769f1-e0cb-4c35-85ba-fe444039ba82', metadata={'source': 'data.txt'}, page_content='User: What is machine learning? , conversation\nBot: Machine learning is a field of AI that allows systems to learn from data. , conversation\nUser: Can you help me with Python? , conversation'),
 Document(id='af3f466f-c969-481c-b601-cafb6ee2efe5', metadata={'source': 'data.txt'}, page_content='User: What is machine learning? , conversation\nBot: Machine learning is a field of AI that allows systems to learn from data. , conversation\nUser: Can you help me with Python? , conversation'),
 Document(id='ac1b0be2-aa95-4b3e-9416-d0eed19bce33', metadata={'source': 'data.txt'}, page_content='User: What is machine learning? , conversation\nBot: Machine learning is a field of AI that allows systems to learn from data. , conversation\nUser: Can you help me with Python? , conversation'),
 Document(id='e39f1308-674f-4e85-b0f1-e324b8e37924', metadata={'source': 'data.txt'}, page_content='User: Hi, ho

In [52]:
vector_db=Chroma.from_documents(docs,embedding,persist_directory="chroma_db")